In [12]:
import boto3
import pandas as pd
import os
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables from .env file (if it exists)
load_dotenv()

# Initialize DynamoDB client
aws_region = os.getenv('AWS_REGION', 'eu-north-1')
endpoint_url = os.getenv('AWS_ENDPOINT_URL')
DYNAMODB_TABLE = os.getenv('DYNAMODB_TABLE', 'apollolytics_dialogues')

if endpoint_url:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region, endpoint_url=endpoint_url)
else:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region)

# Get the table
table = dynamodb.Table(DYNAMODB_TABLE)

# Scan the table to get all items
response = table.scan()
items = response['Items']

# Handle pagination if there are more results
while 'LastEvaluatedKey' in response:
    response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
    items.extend(response['Items'])

# Convert to DataFrame
df = pd.DataFrame(items)

# Convert timestamp to datetime for better analysis
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

# Sort by session_id and timestamp
df = df.sort_values(['session_id', 'timestamp'])

# Display basic info
# print(f"Total records: {len(df)}")
# print(f"Unique sessions: {df['session_id'].nunique()}")
# print(f"\nColumns in the dataset:")
# print(df.columns.tolist())
print(df.shape)
session_ids = df[df.prolific_id.str.len() == 24].session_id
df = df[df.session_id.isin(session_ids)]
print(df.shape)
df = df.dropna(axis=1, how='all')
print(df.shape) 

# Forward fill session-specific columns within each session
session_columns = ['dialogue_mode', 'origin_url', 'article', 'prolific_id']

# Sort by session_id and timestamp to ensure proper order
df = df.sort_values(['session_id', 'timestamp'])

# Forward fill the session columns within each session
df[session_columns] = df.groupby('session_id')[session_columns].ffill()

# Display first few rows
df.head()

(1387, 17)
(441, 17)
(441, 15)


,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,role,message_id,reason,prolific_id,propaganda_result,content,timing_info,datetime
499,session_init,2025-07-16T07:12:53.884060,006808d6-22a1-48b9-bbb8-612607ae6f1a,1.752650e+09,supportive,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,NaN,NaN,NaN,5baa21188a01680001ff1a6f,NaN,NaN,NaN,2025-07-16 07:12:53
500,propaganda_analysis,2025-07-16T07:12:53.960803,006808d6-22a1-48b9-bbb8-612607ae6f1a,1.752650e+09,supportive,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,NaN,NaN,NaN,5baa21188a01680001ff1a6f,"{'type': 'contextualization', 'data': {'Poor J...",NaN,NaN,2025-07-16 07:12:54
501,message,2025-07-16T07:13:07.881367,006808d6-22a1-48b9-bbb8-612607ae6f1a,1.752650e+09,supportive,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,assistant,assistant_6fbc3721-9720-473b-b1f2-28ed9e5b77ac,NaN,5baa21188a01680001ff1a6f,NaN,Certainly! Let's dive into the article's main ...,"{'model_generation_time': 13.911908149719238, ...",2025-07-16 07:13:07
502,message,2025-07-16T07:14:42.802635,006808d6-22a1-48b9-bbb8-612607ae6f1a,1.752650e+09,supportive,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,user,user_bd545aec-cb4f-459f-a0b3-bc496639bc30,NaN,5baa21188a01680001ff1a6f,NaN,you,"{'thinking_time': 29.019, 'recording_duration'...",2025-07-16 07:14:42
503,message,2025-07-16T07:15:00.713847,006808d6-22a1-48b9-bbb8-612607ae6f1a,1.752650e+09,supportive,https://apollolytics-dialogue.vercel.app/dialo...,Trump admin asking federal agencies to cancel ...,assistant,assistant_4a68a913-4866-49db-b0e0-5f9e66a150d6,NaN,5baa21188a01680001ff1a6f,NaN,The article highlights how the Trump administr...,"{'model_generation_time': 15.97230577468872, '...",2025-07-16 07:15:00


In [13]:
# Count message events per session
conversation_turns = df[df['event_type'] == 'message'].groupby('session_id').size()

# Calculate statistics
avg_turns = conversation_turns.mean()
median_turns = conversation_turns.median()
min_turns = conversation_turns.min()
max_turns = conversation_turns.max()

print(f"Average conversation turns per session: {avg_turns:.2f}")
print(f"Median conversation turns per session: {median_turns:.2f}")
print(f"Min conversation turns per session: {min_turns}")
print(f"Max conversation turns per session: {max_turns}")
print(f"Total sessions: {len(conversation_turns)}")
print(f"Total conversation turns: {conversation_turns.sum()}")

# Optional: Show distribution
print("\nDistribution of conversation turns:")
print(conversation_turns.value_counts().sort_index())

Average conversation turns per session: 9.19
Median conversation turns per session: 5.50
Min conversation turns per session: 1
Max conversation turns per session: 35
Total sessions: 36
Total conversation turns: 331

Distribution of conversation turns:
1     12
2      1
3      3
5      2
6      1
7      1
8      1
10     2
12     1
13     2
14     1
17     1
18     3
22     2
24     1
31     1
35     1
Name: count, dtype: int64


In [6]:
#filter by date newest to oldest
df = df.sort_values(by='timestamp', ascending=False)
# filter for dates 2025-07-14
valid_prolific_ids = df[df['prolific_id'].str.len() == 24].prolific_id.unique()
session_ids = df[df.prolific_id.isin(valid_prolific_ids)].session_id.unique()
df = df.dropna(axis=1, how='all')

In [7]:
df.shape

(1387, 17)

In [8]:
df

,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime
1180,message,2025-07-17T06:11:32.109470,b19cdcee-2b98-45a5-88fb-121630eeace7,1.752733e+09,NaN,NaN,NaN,NaN,assistant,assistant_82fb09b6-b83e-41e2-8bb7-295b9c51e6bf,NaN,NaN,NaN,NaN,Certainly! Let's begin with a broad question. ...,"{'model_generation_time': 9.943742752075195, '...",2025-07-17 06:11:32
1179,propaganda_analysis,2025-07-17T06:11:22.155666,b19cdcee-2b98-45a5-88fb-121630eeace7,1.752733e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'contextualization', 'data': {}, 'use...",NaN,NaN,2025-07-17 06:11:23
1178,session_init,2025-07-17T06:11:20.807633,b19cdcee-2b98-45a5-88fb-121630eeace7,1.752733e+09,critical,https://apollolytics-dialogue.vercel.app/dialo...,hello\n,NaN,NaN,NaN,NaN,NaN,XXX,NaN,NaN,NaN,2025-07-17 06:11:20
966,session_end,2025-07-16T22:16:53.814579,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,client_disconnected,NaN,NaN,NaN,NaN,2025-07-16 22:16:53
965,message,2025-07-16T22:16:20.347260,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,assistant,assistant_2572f545-ecc5-4e92-a307-aeba9aa0e934,NaN,NaN,NaN,NaN,That's a very interesting point. You’ve highli...,"{'model_generation_time': 100.93541693687439, ...",2025-07-16 22:16:20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1379,session_init,2025-03-09T14:31:57.797426,dacc1dcf-3b89-400e-b354-750c5a12f959,1.741531e+09,critical,http://localhost:3000/dialogue/positive,sdafasdfasdf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-03-09 14:31:57
604,message,2025-03-09T14:19:03.071867,d3856a40-7ddf-4584-8fb0-605fefdd9b50,1.741530e+09,NaN,NaN,NaN,"Greetings, Kilian. The statement you've offere...",assistant,assistant_ff75904d-5150-4bb8-9e5e-474f4d0ecc6a,"Greetings, Kilian. The statement you've offere...",NaN,NaN,NaN,NaN,NaN,2025-03-09 14:19:03
603,message,2025-03-09T14:18:24.212834,d3856a40-7ddf-4584-8fb0-605fefdd9b50,1.741530e+09,NaN,NaN,NaN,Certainly! Let's dive into a reflective discus...,assistant,assistant_0f5bbeae-3310-4ac0-ba2f-61a866bb1a93,Certainly! Let's dive into a reflective discus...,NaN,NaN,NaN,NaN,NaN,2025-03-09 14:18:24
602,message,2025-03-09T14:18:15.808658,d3856a40-7ddf-4584-8fb0-605fefdd9b50,1.741530e+09,NaN,NaN,NaN,"[{'type': 'text', 'text': 'Please start the co...",user,user_23ece796-e3a2-4efe-8942-4a4a29b30cce,Please start the conversation.,NaN,NaN,NaN,NaN,NaN,2025-03-09 14:18:15


In [5]:
df[(df.session_id.isin(session_ids))]

,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime
966,session_end,2025-07-16T22:16:53.814579,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,client_disconnected,NaN,NaN,NaN,NaN,2025-07-16 22:16:53
965,message,2025-07-16T22:16:20.347260,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,assistant,assistant_2572f545-ecc5-4e92-a307-aeba9aa0e934,NaN,NaN,NaN,NaN,That's a very interesting point. You’ve highli...,"{'model_generation_time': 100.93541693687439, ...",2025-07-16 22:16:20
964,message,2025-07-16T22:14:37.980889,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,user,user_ed966ef8-6b13-46f2-aa95-1e5ed66c7568,NaN,NaN,NaN,NaN,"Yes, definitely, I think the public concern wo...","{'thinking_time': 26.328, 'recording_duration'...",2025-07-16 22:14:37
963,message,2025-07-16T22:13:22.252938,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,assistant,assistant_f785eb81-90a5-4422-8e18-af12c0cad207,NaN,NaN,NaN,NaN,You’re touching on a really interesting aspect...,"{'model_generation_time': 73.20751476287842, '...",2025-07-16 22:13:22
962,message,2025-07-16T22:12:08.268973,ca20b153-f513-42b3-892e-1592d7337ba0,1.752704e+09,NaN,NaN,NaN,NaN,user,user_08d55ec7-bbe4-4da5-8edf-ea2883a6ad9e,NaN,NaN,NaN,NaN,"I think probably over time, the public's opini...","{'thinking_time': 14.638, 'recording_duration'...",2025-07-16 22:12:08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
912,message,2025-07-15T13:32:40.776477,12f7327e-a8ee-4d69-b49a-fbedc65e0970,1.752586e+09,NaN,NaN,NaN,NaN,assistant,assistant_299b2057-669d-42b9-8de3-36352888a2d0,NaN,NaN,NaN,NaN,That's an interesting perspective. You’re poin...,"{'model_generation_time': 15.687318801879883, ...",2025-07-15 13:32:40
911,message,2025-07-15T13:32:23.669346,12f7327e-a8ee-4d69-b49a-fbedc65e0970,1.752586e+09,NaN,NaN,NaN,NaN,user,user_0e27f39c-ba91-4cea-aa09-7578278b4429,NaN,NaN,NaN,NaN,I think without a doubt there are biases. Cert...,"{'thinking_time': 2.504, 'recording_duration':...",2025-07-15 13:32:23
910,message,2025-07-15T13:31:34.203553,12f7327e-a8ee-4d69-b49a-fbedc65e0970,1.752586e+09,NaN,NaN,NaN,NaN,assistant,assistant_04798519-267a-4e46-9b56-a1154a7e0efe,NaN,NaN,NaN,NaN,What do you think about the administration’s m...,"{'model_generation_time': 6.112159013748169, '...",2025-07-15 13:31:34
909,propaganda_analysis,2025-07-15T13:31:28.082875,12f7327e-a8ee-4d69-b49a-fbedc65e0970,1.752586e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'type': 'contextualization', 'data': {'Poor J...",NaN,NaN,2025-07-15 13:31:29


In [90]:
import pandas as pd

# Filter the DataFrame as you did
filtered = df[(df.session_id.isin(session_ids)) & (df.role == 'user')]

# Extract timing_info dicts into a DataFrame
timing_df = pd.json_normalize(filtered['timing_info'].dropna())

# Convert all columns to float (in case they are Decimal or string)
timing_df = timing_df.apply(pd.to_numeric, errors='coerce')

# Calculate the mean for each timing field
means = timing_df.mean()

print(means)

thinking_time          26.495381
recording_duration     26.205750
total_response_time    52.701131
dtype: float64


In [91]:
timing_df

,thinking_time,recording_duration,total_response_time
0,16.578,28.509,45.087
1,47.138,20.232,67.370
2,4.100,26.744,30.844
3,29.019,19.019,48.038
4,64.232,49.481,113.713
...,...,...,...
79,4.025,60.682,64.707
80,14.673,68.407,83.080
81,53.318,52.159,105.477
82,6.057,49.906,55.963


In [8]:
#!/usr/bin/env python3
"""
Debug script to query DynamoDB and show all events for a specific session.
"""

import boto3
import os
import json
from datetime import datetime

# AWS configuration
aws_region = os.environ.get('AWS_REGION', 'eu-north-1')
DYNAMODB_TABLE = os.environ.get('DYNAMODB_TABLE', 'apollolytics_dialogues')

# Initialize DynamoDB
dynamodb = boto3.resource('dynamodb', region_name=aws_region)
table = dynamodb.Table(DYNAMODB_TABLE)

def query_session_data(session_id: str):
    """Query all data for a specific session."""
    try:
        response = table.query(
            KeyConditionExpression=boto3.dynamodb.conditions.Key('session_id').eq(session_id)
        )
        
        items = response.get('Items', [])
        print(f"\n=== Session Data for {session_id} ===")
        print(f"Total items found: {len(items)}")
        
        for i, item in enumerate(items):
            print(f"\n--- Item {i+1} ---")
            print(f"Session ID: {item.get('session_id')}")
            print(f"Timestamp: {item.get('timestamp')}")
            print(f"Event Type: {item.get('event_type', 'MISSING!')}")
            print(f"Created At: {item.get('created_at')}")
            
            # Show specific fields based on event type
            event_type = item.get('event_type')
            if event_type == 'session_init':
                print(f"Dialogue Mode: {item.get('dialogue_mode')}")
                print(f"Origin URL: {item.get('origin_url')}")
                print(f"Prolific ID: {item.get('prolific_id')}")
                print(f"Article Length: {len(item.get('article', ''))} chars")
            elif event_type == 'propaganda_analysis':
                print(f"Propaganda Result Keys: {list(item.get('propaganda_result', {}).keys())}")
            elif event_type == 'session_end':
                print(f"Reason: {item.get('reason')}")
            else:
                # Message event (no event_type field)
                print(f"Role: {item.get('role')}")
                print(f"Message ID: {item.get('message_id')}")
                print(f"Content: {item.get('content', '')}...")
                print(f"Timing Info: {item.get('timing_info', {})}")
            
            print("-" * 50)
            
        return items
        
    except Exception as e:
        print(f"Error querying session data: {e}")
        return []

def list_recent_sessions(limit=10):
    """List recent sessions."""
    try:
        response = table.scan(
            ProjectionExpression="session_id, #ts, event_type, created_at",
            ExpressionAttributeNames={"#ts": "timestamp"},
            Limit=limit
        )
        
        items = response.get('Items', [])
        print(f"\n=== Recent Sessions (showing up to {limit}) ===")
        
        # Group by session_id and show event types
        sessions = {}
        for item in items:
            session_id = item.get('session_id')
            event_type = item.get('event_type', 'message')
            created_at = item.get('created_at')
            
            if session_id not in sessions:
                sessions[session_id] = {'events': [], 'created_at': created_at}
            sessions[session_id]['events'].append(event_type)
        
        for session_id, data in sessions.items():
            events = data['events']
            created_at = data['created_at']
            print(f"Session: {session_id}")
            print(f"  Created: {created_at}")
            print(f"  Events: {events}")
            print()
            
    except Exception as e:
        print(f"Error listing sessions: {e}")

if __name__ == "__main__":
    # List recent sessions first
    list_recent_sessions(5)
    
    # Query specific session (use the session ID from your logs)
    session_id = "8a52bbf1-2381-44e4-b682-9407be72896c"  # From your logs
    query_session_data(session_id) 


=== Recent Sessions (showing up to 5) ===
Session: 24399df5-600c-4db8-a3e6-a051904ae466
  Created: 2025-05-07T10:14:55.153239
  Events: ['session_init', 'message', 'message', 'message', 'session_end']


=== Session Data for 8a52bbf1-2381-44e4-b682-9407be72896c ===
Total items found: 3

--- Item 1 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 1752580559020550
Event Type: session_init
Created At: 2025-07-15T11:55:59.020553
Dialogue Mode: critical
Origin URL: https://apollolytics-dialogue.vercel.app/dialogue/positive1
Prolific ID: KDOT
Article Length: 5652 chars
--------------------------------------------------

--- Item 2 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 1752580559035008
Event Type: propaganda_analysis
Created At: 2025-07-15T11:55:59.035195
Propaganda Result Keys: ['type', 'data', 'user_id', 'status']
--------------------------------------------------

--- Item 3 ---
Session ID: 8a52bbf1-2381-44e4-b682-9407be72896c
Timestamp: 17525805658

In [ ]:
# Debug script to check what's actually in the database
import boto3
import pandas as pd
import os
from datetime import datetime

# Load environment variables
aws_region = os.getenv('AWS_REGION', 'eu-north-1')
endpoint_url = os.getenv('AWS_ENDPOINT_URL')
DYNAMODB_TABLE = os.getenv('DYNAMODB_TABLE', 'apollolytics_dialogues')

if endpoint_url:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region, endpoint_url=endpoint_url)
else:
    dynamodb = boto3.resource('dynamodb', region_name=aws_region)

table = dynamodb.Table(DYNAMODB_TABLE)

# Get the most recent session_init events
response = table.scan(
    FilterExpression=boto3.dynamodb.conditions.Attr('event_type').eq('session_init'),
    Limit=10
)

items = response['Items']

# Handle pagination
while 'LastEvaluatedKey' in response and len(items) < 10:
    response = table.scan(
        FilterExpression=boto3.dynamodb.conditions.Attr('event_type').eq('session_init'),
        ExclusiveStartKey=response['LastEvaluatedKey'],
        Limit=10-len(items)
    )
    items.extend(response['Items'])

# Convert to DataFrame
df_debug = pd.DataFrame(items)

if len(df_debug) > 0:
    print("Most recent session_init events:")
    print(df_debug[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'created_at']].to_string())
    
    print(f"\nData types:")
    print(df_debug.dtypes)
    
    print(f"\nUnique prolific_ids:")
    print(df_debug['prolific_id'].unique())
    
    print(f"\nUnique dialogue_modes:")
    print(df_debug['dialogue_mode'].unique())
    
    print(f"\nUnique origin_urls:")
    print(df_debug['origin_url'].unique())
else:
    print("No session_init events found in database")

Most recent session_init events:
                             session_id prolific_id dialogue_mode                                                  origin_url                  created_at
0  24399df5-600c-4db8-a3e6-a051904ae466         NaN      critical  https://apollolytics-dialogue.vercel.app/dialogue/positive  2025-05-07T10:14:55.153239
1  21aed941-b5ba-4a94-943e-557478325fb4         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-14T14:48:16.686450
2  54aab53e-bedb-4a84-aa3b-14557c90d5e2         XXX      critical                     http://localhost:3000/dialogue/positive  2025-05-26T09:44:39.566374
3  083ef2d2-f651-42af-aaf7-24ab6b26dfa1         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-19T15:08:58.871177
4  8028699e-b7de-4e1e-8b50-cd8c46c9194e         NaN      critical                     http://localhost:3000/dialogue/positive  2025-05-19T17:36:30.020041
5  9477a8c5-0ad2-4ec2-966e-3387986fce42    

In [32]:
# Filter for session_init events before 2025-07-13
cutoff_date = pd.to_datetime("2025-07-13")

df_before_cutoff = df[
    (df['event_type'] == 'session_init') &
    (df['datetime'] < cutoff_date)
]

print(f"Session_init events before 2025-07-13: {len(df_before_cutoff)}")
print(df_before_cutoff[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())
# Example: Filter for experiment sessions with valid prolific IDs before cutoff date
experiment_sessions = df[
    (df['event_type'] == 'session_init') &
    (df['prolific_id'].str.len() == 24) &
    (df['origin_url'].str.contains('positive[123]|negative[123]', na=False)) &
    (df['datetime'] < cutoff_date)
]['session_id'].unique()

df_experiment = df[df['session_id'].isin(experiment_sessions)].copy()
df_experiment = df_experiment.sort_values(['datetime', 'session_id', 'timestamp'], ascending=[False, True, True])

print(f"Filtered experiment dataset before 2025-07-13:")
print(df_experiment[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())

Session_init events before 2025-07-13: 0
Empty DataFrame
Columns: [session_id, prolific_id, dialogue_mode, origin_url, datetime]
Index: []
Filtered experiment dataset before 2025-07-13:
Empty DataFrame
Columns: [session_id, prolific_id, dialogue_mode, origin_url, datetime]
Index: []


In [3]:
# Filter for sessions with valid prolific IDs (24 characters long)
valid_prolific_sessions = df[
    (df['event_type'] == 'session_init') & 
    (df['prolific_id'].str.len() == 24)
]['session_id'].unique()

print(f"Found {len(valid_prolific_sessions)} sessions with valid prolific IDs")

# Filter the main dataframe to only include sessions with valid prolific IDs
df_valid_prolific = df[df['session_id'].isin(valid_prolific_sessions)].copy()

# Sort by session_id and timestamp
df_valid_prolific = df_valid_prolific.sort_values(['session_id', 'timestamp'])

# Display summary
print(f"\nFiltered dataset:")
print(f"Total records: {len(df_valid_prolific)}")
print(f"Unique sessions: {df_valid_prolific['session_id'].nunique()}")

# Show unique prolific IDs
valid_prolific_ids = df_valid_prolific[
    (df_valid_prolific['event_type'] == 'session_init') & 
    (df_valid_prolific['prolific_id'].str.len() == 24)
]['prolific_id'].unique()

print(f"\nValid prolific IDs found:")
for prolific_id in valid_prolific_ids:
    print(f"  - {prolific_id}")

# Display first few rows of filtered data
df_valid_prolific.head(10)

Found 0 sessions with valid prolific IDs

Filtered dataset:
Total records: 0
Unique sessions: 0

Valid prolific IDs found:


,event_type,created_at,session_id,timestamp,dialogue_mode,origin_url,article,message_content,role,message_id,transcript,reason,prolific_id,propaganda_result,content,timing_info,datetime


In [26]:
# Check ALL prolific IDs in the database, regardless of length
all_prolific_ids = df[
    (df['event_type'] == 'session_init') & 
    (df['prolific_id'].notna())
]['prolific_id'].unique()

print(f"All prolific IDs found (any length):")
for prolific_id in all_prolific_ids:
    print(f"  - '{prolific_id}' (length: {len(prolific_id)})")

# Check for any recent sessions from experiment routes
recent_experiment_sessions = df[
    (df['event_type'] == 'session_init') & 
    (df['origin_url'].str.contains('positive[123]|negative[123]', na=False))
]

print(f"\nRecent experiment sessions:")
if len(recent_experiment_sessions) > 0:
    print(recent_experiment_sessions[['session_id', 'prolific_id', 'origin_url', 'datetime']].to_string())
else:
    print("No experiment sessions found")

# Check the most recent sessions overall
recent_sessions = df[
    df['event_type'] == 'session_init'
].sort_values('datetime', ascending=False).head(5)

print(f"\nMost recent 5 sessions:")
print(recent_sessions[['session_id', 'prolific_id', 'dialogue_mode', 'origin_url', 'datetime']].to_string())

All prolific IDs found (any length):
  - 'XXX' (length: 3)
  - 'saf' (length: 3)

Recent experiment sessions:
                               session_id prolific_id                                                   origin_url            datetime
445  56a30a89-698d-4161-83a9-bf8ea76aa7d9         saf  https://apollolytics-dialogue.vercel.app/dialogue/negative3 2025-07-13 16:41:05

Most recent 5 sessions:
                               session_id prolific_id dialogue_mode                                                   origin_url            datetime
312  3924414e-e212-48c8-a4bd-b0f61b2532ff         XXX      critical   https://apollolytics-dialogue.vercel.app/dialogue/positive 2025-07-13 19:56:14
445  56a30a89-698d-4161-83a9-bf8ea76aa7d9         saf    supportive  https://apollolytics-dialogue.vercel.app/dialogue/negative3 2025-07-13 16:41:05


In [7]:
df.origin_url.unique()

array(['http://localhost:3000/dialogue/positive', nan,
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/positive1',
       'https://apollolytics-dialogue.vercel.app/dialogue/negative',
       'https://apollolytics-dialogue.vercel.app/dialogue/positive',
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/negative1',
       'https://apollolytics-dialogue.vercel.app/dialogue/negative3',
       'https://apollolytics-dialogue.vercel.app/dialogue/positive1',
       'https://apollolytics-dialogue-kxqo4hr1x-kilian-sprenkamps-projects.vercel.app/dialogue/positive3',
       'http://localhost:3000/dialogue/positive2'], dtype=object)